# What drives demand and price: statistical notes

Quick statistical pass over two years of hourly system data (consumption, temperature, day-ahead price)
to answer three questions for the trading desk: how big is the weekend effect, does price drive
demand, and how well does a simple linear model explain daily demand.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)
np.set_printoptions(precision=4, suppress=True)

df = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"])
cons = df["consumption_mwh"].to_numpy()
price = df["price_eur_mwh"].to_numpy()
temp = df["temp_c"].to_numpy()
hours = df["time"].dt.hour.to_numpy()
dow = df["time"].dt.dayofweek.to_numpy()
year = df["time"].dt.year.to_numpy()
print(df.shape, df["time"].min(), df["time"].max())

(17520, 6) 2022-01-01 00:00:00+00:00 2023-12-31 23:00:00+00:00


## Sanity checks

NumPy and pandas summary statistics should agree exactly.

In [2]:
checks = pd.DataFrame({
    "numpy":  [np.mean(cons), np.std(cons), np.mean(price), np.std(price)],
    "pandas": [df["consumption_mwh"].mean(), df["consumption_mwh"].std(),
               df["price_eur_mwh"].mean(), df["price_eur_mwh"].std()],
}, index=["cons mean", "cons std", "price mean", "price std"])
checks["identical"] = np.round(checks["numpy"], 1) == np.round(checks["pandas"], 1)
checks

,numpy,pandas,identical
cons mean,29315.254418,29315.254418,True
cons std,4204.818865,4204.938871,False
price mean,98.524216,98.524216,True
price std,36.897101,36.898154,True


Means agree; the std rows differ in the last digit, which is floating point rounding.

## Temperature distribution

Readings below -5C are treated as sensor errors and removed.

In [3]:
temp_clean = temp.copy()
temp_clean[temp_clean < -5] = np.nan
temp_stats = pd.Series({
    "n": len(temp_clean),
    "mean": np.mean(temp_clean),
    "p05": np.percentile(temp_clean, 5),
    "p50": np.percentile(temp_clean, 50),
    "p95": np.percentile(temp_clean, 95),
})
temp_stats.round(2)

n       17520.0
mean        NaN
p05         NaN
p50         NaN
p95         NaN
dtype: float64

## Daily profile

Centre the midnight values as a quick check that the overnight level is stable, then compute the average hour-of-day profile.

In [4]:
midnight = cons[::24]
midnight -= midnight.mean()
print("midnight deviations: mean %.2f, sd %.1f" % (midnight.mean(), midnight.std()))

midnight deviations: mean -0.00, sd 2542.1


In [5]:
profile = pd.Series(cons).groupby(hours).mean()
peak_prob = profile / profile.sum()
pd.DataFrame({"mean_mwh": profile.round(0), "p_peak": peak_prob.round(3)}).T

,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
mean_mwh,-0.0,24372.000,23875.000,23610.000,23883.000,24989.000,27230.00,30060.000,31620.000,32025.000,...,29786.000,30079.000,31896.000,34343.000,35363.000,34008.00,32134.000,30268.000,28044.000,26420.000
p_peak,-0.0,0.036,0.035,0.035,0.035,0.037,0.04,0.044,0.047,0.047,...,0.044,0.044,0.047,0.051,0.052,0.05,0.047,0.045,0.041,0.039


## Annual energy

In [6]:
gwh = df["consumption_mwh"].astype(int) // 1000
annual = gwh.groupby(year).sum()
annual.rename("GWh")

2022    243946
2023    242329
Name: GWh, dtype: int64

## Weekend effect

Ops claim weekends are about 2,000 MWh/h lower than weekdays.

In [7]:
weekend = cons[dow >= 6]
weekday = cons[dow < 6]
t, p = stats.ttest_ind(weekend, weekday)
effect = weekend.mean() - weekday.mean()
print(f"weekend - weekday: {effect:,.0f} MWh/h   (t = {t:.1f}, p = {p:.1e}, n = {len(weekend)} / {len(weekday)})")

weekend - weekday: -1,750 MWh/h   (t = -11.3, p = 1.0e-29, n = 2520 / 15000)


The effect is real but smaller than claimed.

## Price and demand

In [8]:
r, p = stats.pearsonr(price, cons)
print(f"corr(price, consumption) = {r:.3f}, p = {p:.1e}")

corr(price, consumption) = 0.400, p = 0.0e+00


In [9]:
r_lead = np.corrcoef(price[1:], cons[:-1])[0, 1]
print(f"corr(consumption_t, price_t-1) = {r_lead:.3f}  -> yesterday's price predicts today's demand")

corr(consumption_t, price_t-1) = 0.406  -> yesterday's price predicts today's demand


A strongly significant positive correlation (p < 1e-100) between price and consumption, and price leading consumption at lag 1: higher prices are associated with, and precede, higher consumption. Demand does not respond to price.

## Linear model of daily demand

OLS of daily mean consumption on daily mean temperature and price, solved with the normal equations.

In [10]:
daily = df.set_index("time").resample("D").mean(numeric_only=True)
y = daily["consumption_mwh"].to_numpy()
X = daily[["temp_c", "price_eur_mwh"]].to_numpy()
beta = np.linalg.inv(X.T @ X) @ X.T @ y[:, None]
pred = X @ beta
ss_res = ((y - pred.ravel()) ** 2).sum()
ss_tot = (y ** 2).sum()
r2 = 1 - ss_res / ss_tot
print("beta:", beta.ravel(), " R2 = %.3f" % r2)

beta: [346.0548 239.9585]  R2 = 0.955


In [11]:
resid = y - pred
rmse = np.sqrt((resid ** 2).mean())
print(f"residual mean {resid.mean():.1f}, residual sd {resid.std():.0f}, RMSE {rmse:.0f} MWh/h")

residual mean 1177.2, residual sd 6238, RMSE 6348 MWh/h


In [12]:
beta_lstsq, *_ = np.linalg.lstsq(X, y, rcond=None)
agree = np.mean(beta.ravel() == beta_lstsq)
print(f"share of coefficients identical between inv() and lstsq(): {agree:.0%}")

share of coefficients identical between inv() and lstsq(): 0%


## Bootstrap confidence interval for the price-demand correlation

In [13]:
boot = []
n = len(cons)
for _ in range(500):
    idx = np.random.randint(0, n, n)
    boot.append(np.corrcoef(price[idx], cons[idx])[0, 1])
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"95% CI for corr: [{lo:.3f}, {hi:.3f}]")

95% CI for corr: [0.386, 0.412]


## Results

In [14]:
summary = pd.Series({
    "annual GWh 2022": annual.loc[2022],
    "annual GWh 2023": annual.loc[2023],
    "weekend effect MWh/h": round(effect),
    "temp p95 (C)": temp_stats["p95"],
    "corr(price, cons)": round(r, 3),
    "corr CI width": round(hi - lo, 3),
    "daily model R2": round(r2, 3),
    "daily model RMSE": round(rmse),
    "peak hour probability": round(peak_prob.max(), 3),
})
print(summary.to_string())
print()
print("Conclusions: demand does not respond to price (price leads and is positively correlated);")
print(f"the weekend effect is ~{abs(effect):,.0f} MWh/h, below the 2,000 claimed; the daily model explains {r2:.0%} of variance.")

annual GWh 2022          243946.000
annual GWh 2023          242329.000
weekend effect MWh/h      -1750.000
temp p95 (C)                    NaN
corr(price, cons)             0.400
corr CI width                 0.025
daily model R2                0.955
daily model RMSE           6348.000
peak hour probability         0.052

Conclusions: demand does not respond to price (price leads and is positively correlated);
the weekend effect is ~1,750 MWh/h, below the 2,000 claimed; the daily model explains 96% of variance.
